# Lab 01: Document Parsing & Chunking

## Overview
Parse arXiv AI/ML papers stored in a Unity Catalog Volume into structured data, then chunk the content for downstream vector search indexing.

## Learning Objectives
- Use `ai_parse_document()` to extract text, tables, and images from PDFs
- Compare SQL and Python parsing approaches
- Clean parsed content using `ai_query()` for semantic cleaning
- Implement recursive character chunking with configurable overlap
- Save flattened chunks to a Delta table ready for embedding

## Exam Domain
**Data Preparation (14%)** — This lab covers document ingestion, parsing, transformation, and chunking — core skills tested in the Data Preparation section.

In [ ]:
# Prerequisites — run once per cluster
%pip install databricks-sdk mlflow --quiet
dbutils.library.restartPython()

In [ ]:
# Configuration — shared across all labs
CATALOG = "genai_lab_guide"
SCHEMA = "default"
VOLUME = "arxiv_papers"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

# Verify the volume exists and list papers
display(spark.sql(f"LIST '{VOLUME_PATH}'"))

## A. Parsing Documents with Python

We use `ai_parse_document()` to extract structured content from PDFs. This Databricks SQL function leverages Mosaic AI to detect text, tables, and images automatically.

The Python approach uses `expr()` to call the SQL function from a Spark DataFrame.

In [ ]:
from pyspark.sql.functions import expr

# Read all PDFs as binary files
docs_df = spark.read.format("binaryFile").load(VOLUME_PATH)

print(f"Found {docs_df.count()} documents")
display(docs_df.select("path", "length"))

In [ ]:
# Parse each document using ai_parse_document
parsed_df = docs_df.withColumn(
    "parsed_content",
    expr("ai_parse_document(content, map('version', '2.0'))")
)

display(parsed_df.select("path", "parsed_content"))

## B. Parsing with SQL

The same operation in pure SQL — ideal for batch processing and scheduled jobs.

In [ ]:
parsed_sql_df = spark.sql(f"""
SELECT
  path,
  ai_parse_document(
    content,
    map('version', '2.0')
  ) AS parsed_doc
FROM read_files('{VOLUME_PATH}', format => 'binaryFile')
""")

display(parsed_sql_df)

## C. Inspecting Parsed Output

The `ai_parse_document` output contains rich metadata:
- **`document:pages`** — list of page objects with `page_number`, `text`, `tables`
- **`document:elements`** — individual elements (paragraphs, headings, tables, images)
- **`error_status`** — any parsing errors
- **`metadata`** — document-level metadata

In [ ]:
from pyspark.sql.functions import expr, col

# Extract key metadata fields
meta_df = parsed_df.select(
    "path",
    expr("parsed_content:document:pages"),
    expr("parsed_content:document:elements"),
    expr("parsed_content:error_status"),
    expr("parsed_content:metadata")
)

display(meta_df)

In [ ]:
# Extract text from all pages of the first document
from pyspark.sql.functions import explode, col

pages_df = parsed_df.select(
    "path",
    explode(expr("parsed_content:document:pages")).alias("page")
).select(
    "path",
    col("page.page_number").alias("page_number"),
    col("page.text").alias("page_text")
)

display(pages_df)

## D. Semantic Cleaning with ai_query()

Raw parsed text often contains artifacts: broken lines, orphaned headers, inconsistent spacing. Using `ai_query()` with a Foundation Model, we can clean the text semantically — preserving structure while removing noise.

This is higher quality than simple regex/concatenation but costs more (LLM tokens).

In [ ]:
from pyspark.sql.functions import concat_ws, collect_list

# Concatenate all page text per document
raw_text_df = pages_df.groupBy("path").agg(
    concat_ws("\n\n", collect_list("page_text")).alias("raw_text")
)

# Use ai_query() for semantic cleaning
cleaned_df = raw_text_df.withColumn(
    "cleaned_text",
    expr("""ai_query(
        'databricks-meta-llama-3-1-8b-instruct',
        CONCAT(
            'Clean the following academic paper text. Remove artifacts like broken lines, ',
            'orphaned headers, and page numbers. Preserve the original content, section structure, ',
            'and meaning exactly. Return only the cleaned text, nothing else.\n\n',
            raw_text
        )
    )""")
)

display(cleaned_df.select("path", "cleaned_text"))

## E. Chunking Documents

We split the cleaned text into chunks suitable for embedding and vector search. Key parameters:
- **chunk_size**: Maximum tokens per chunk (512 is a common choice)
- **chunk_overlap**: Tokens shared between adjacent chunks (prevents context loss at boundaries)

We use recursive character splitting, which tries to split on paragraph boundaries first, then sentences, then words.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from pyspark.sql.functions import udf, posexplode, monotonically_increasing_id, lit, sha2, concat
from pyspark.sql.types import ArrayType, StringType

# Configure chunking
CHUNK_SIZE = 1000       # characters (roughly 250 tokens)
CHUNK_OVERLAP = 200     # characters of overlap between chunks

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# UDF to split text into chunks
@udf(returnType=ArrayType(StringType()))
def split_into_chunks(text):
    if text is None:
        return []
    return text_splitter.split_text(text)

# Apply chunking
chunked_df = cleaned_df.withColumn("chunks", split_into_chunks("cleaned_text"))

# Flatten: one row per chunk with a unique ID
chunks_flat_df = (
    chunked_df
    .select("path", posexplode("chunks").alias("chunk_index", "chunk_text"))
    .withColumn(
        "chunk_id",
        sha2(concat(col("path"), lit("_"), col("chunk_index").cast("string")), 256)
    )
    .select("chunk_id", "path", "chunk_index", "chunk_text")
)

print(f"Total chunks: {chunks_flat_df.count()}")
display(chunks_flat_df)

## F. Save to Delta Table

Save chunks to a Delta table with:
- **Unique ID per row** (`chunk_id`) — required for Vector Search indexing
- **Change Data Feed enabled** — required for Delta Sync Vector Search

In [ ]:
TABLE_NAME = f"{CATALOG}.{SCHEMA}.arxiv_chunks"

# Enable Change Data Feed and write
(
    chunks_flat_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Saved {spark.table(TABLE_NAME).count()} chunks to {TABLE_NAME}")
display(spark.table(TABLE_NAME).limit(5))

## Key Takeaways

1. `ai_parse_document()` extracts text, tables, and images from PDFs — available in both SQL and Python
2. `ai_query()` with a Foundation Model can clean parsed text semantically (higher quality, higher cost)
3. Recursive character splitting preserves semantic boundaries when chunking
4. Each chunk needs a unique ID and the Delta table needs Change Data Feed for Vector Search sync
5. Chunk size and overlap are tunable parameters — smaller chunks = more precise retrieval, larger = more context

See [workbook.md](workbook.md) for architecture diagrams, exam tips, and detailed walkthrough.